# Fold 1: evaluación local (sin FUSE) de baseline y v1 (λ = 0.2) y comparación pareada

Los entrenamientos del fold 1 (baseline y v1 con λ = 0.2, 100 épocas cada uno) están completos y sus predicciones de validación (200 casos por modelo) se encuentran guardadas en Drive. Este notebook **no entrena**: solo evalúa.

La lectura en directo de las predicciones y del ground truth desde Drive montado (FUSE) resultó inestable (procesos bloqueados a mitad de la evaluación, con un `%wa` —espera de disco— elevado en `top`), el mismo patrón de inestabilidad de FUSE observado en las copias masivas. Para evitar FUSE en la parte costosa, se descargan las ≈ 400 imágenes necesarias (200 predicciones y 200 ground truth por modelo) al disco local (`/content`) mediante la **API de Google Drive por HTTP** (con el mismo enfoque de descarga vía API empleado en un intento previo de evaluación), con timeout real y reintentos. Una vez descargadas, toda la evaluación (Dice, clDice, componentes y HD95) se realiza sobre archivos locales, sin dependencia de Drive durante el bucle largo.

Al requerir solo CPU, se utiliza un runtime **sin acelerador**.

Además, el notebook incluye:
- Checkpoint y reanudación ante reinicios del runtime.
- Comparación pareada (Wilcoxon y t-test) en el fold 1.
- Tabla de consistencia entre el fold 0 y el fold 1.


## 1. Autenticación con la API de Drive

Drive se monta también mediante FUSE, pero únicamente para operaciones ligeras (lectura y escritura de CSV pequeños) y nunca para las ≈ 400 imágenes de la evaluación.


In [1]:
from google.colab import auth
auth.authenticate_user()

import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

creds, _ = google.auth.default()
drive_api = build("drive", "v3", credentials=creds)
print("Api de Drive autenticada.")

from google.colab import drive
drive.mount("/content/drive", force_remount=True)
print("Drive montado (solo para CSVs pequenos, no para las imagenes).")

# El timeout de socket se activa después de autenticar (si se activa antes
# puede interferir con la propagación de credenciales de Colab)
import socket
socket.setdefaulttimeout(90)
print("Timeout de socket (90s) activado para las descargas.")


Api de Drive autenticada.
Mounted at /content/drive
Drive montado (solo para CSVs pequenos, no para las imagenes).
Timeout de socket (90s) activado para las descargas.


## 2. Configuración


In [2]:
import os, io, time, shutil
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM_nnUNet")

DATASET_FOLDER_NAME = "Dataset503_ImageCAS1000"
CONFIGURATION = "3d_fullres"
FOLD = 1

BASELINE_TRAINER = "nnUNetTrainer_100epochs"
V1_TRAINER = "nnUNetTrainerV1_1000casos_100epochs"

# Carpetas locales donde se descargan las ≈ 400 imágenes necesarias
LOCAL_EVAL = Path("/content/eval_fold1")
LOCAL_VAL = {
    BASELINE_TRAINER: LOCAL_EVAL / "val_baseline",
    V1_TRAINER: LOCAL_EVAL / "val_v1",
}
LOCAL_GT = LOCAL_EVAL / "gt_segmentations"

print("BASE (Drive, solo CSVs):", BASE)
print("Local eval (todo lo pesado va aqui):", LOCAL_EVAL)


BASE (Drive, solo CSVs): /content/drive/MyDrive/TFM_nnUNet
Local eval (todo lo pesado va aqui): /content/eval_fold1


## 3. Funciones de navegación y descarga de Drive mediante API (sin FUSE)


In [3]:
def buscar_id(nombre, parent_id, carpeta=True):
    mime = " and mimeType = 'application/vnd.google-apps.folder'" if carpeta else ""
    q = f"name = '{nombre}' and '{parent_id}' in parents and trashed = false{mime}"
    res = drive_api.files().list(q=q, fields="files(id, name)", pageSize=5).execute()
    archivos = res.get("files", [])
    if not archivos:
        raise FileNotFoundError(f"No se encontro '{nombre}' dentro de {parent_id}")
    return archivos[0]["id"]

def listar_hijos(parent_id):
    archivos = []
    page_token = None
    while True:
        res = drive_api.files().list(
            q=f"'{parent_id}' in parents and trashed = false",
            fields="nextPageToken, files(id, name, size, mimeType)",
            pageSize=1000,
            pageToken=page_token,
        ).execute()
        archivos.extend(res.get("files", []))
        page_token = res.get("nextPageToken")
        if not page_token:
            break
    return archivos

def descargar_lista(archivos_remotos, carpeta_local, max_reintentos=5):
    carpeta_local.mkdir(parents=True, exist_ok=True)
    print(f"Total de archivos a descargar: {len(archivos_remotos)}")

    pendientes = list(archivos_remotos)
    intento = 0
    while pendientes and intento < max_reintentos:
        intento += 1
        print(f"--- Intento {intento}/{max_reintentos}: {len(pendientes)} archivos pendientes ---")
        fallidos = []
        for i, meta in enumerate(pendientes):
            destino = carpeta_local / meta["name"]
            tam_esperado = int(meta.get("size", 0) or 0)
            try:
                if destino.exists() and (tam_esperado == 0 or destino.stat().st_size == tam_esperado):
                    continue
                request = drive_api.files().get_media(fileId=meta["id"])
                fh = io.FileIO(destino, "wb")
                downloader = MediaIoBaseDownload(fh, request, chunksize=1024 * 1024 * 20)
                done = False
                while not done:
                    status, done = downloader.next_chunk(num_retries=3)
                fh.close()
            except Exception as e:
                fallidos.append(meta)
                if len(fallidos) % 50 == 1:
                    print(f"  fallo en {meta['name']}: {type(e).__name__}: {e}")
            if i % 50 == 0:
                print(f"  {i}/{len(pendientes)} procesados ({len(fallidos)} fallos hasta ahora)")

        pendientes = fallidos
        if pendientes:
            print(f"Quedan {len(pendientes)} archivos fallidos. Esperando 15s antes de reintentar...")
            time.sleep(15)

    if pendientes:
        print(f"ATENCION: tras {max_reintentos} intentos siguen fallando {len(pendientes)} archivos.")
    else:
        print("Descarga completa (verificada por tamano).")
    return pendientes

print("Funciones de Drive API listas.")


Funciones de Drive API listas.


## 4. Localización de las carpetas de validación y del ground truth en Drive

Solo se leen metadatos (nombres, identificadores y tamaños) mediante la API; todavía no se descarga nada.


In [4]:
id_tfm = buscar_id("TFM_nnUNet", "root")
id_results = buscar_id("nnUNet_results", id_tfm)
id_dataset_res = buscar_id(DATASET_FOLDER_NAME, id_results)

archivos_val_por_trainer = {}
for trainer in (BASELINE_TRAINER, V1_TRAINER):
    id_trainer_cfg = buscar_id(f"{trainer}__nnUNetPlans__{CONFIGURATION}", id_dataset_res)
    id_fold = buscar_id(f"fold_{FOLD}", id_trainer_cfg)
    id_val = buscar_id("validation", id_fold)
    archivos_val = listar_hijos(id_val)
    archivos_val_nii = [f for f in archivos_val if f["name"].endswith(".nii.gz")]
    archivos_val_por_trainer[trainer] = archivos_val_nii
    print(f"{trainer}: {len(archivos_val_nii)} predicciones .nii.gz encontradas en Drive.")

id_prep = buscar_id("nnUNet_preprocessed", id_tfm)
id_dataset_prep = buscar_id(DATASET_FOLDER_NAME, id_prep)
id_gt = buscar_id("gt_segmentations", id_dataset_prep)
archivos_gt_todos = listar_hijos(id_gt)
print(f"gt_segmentations: {len(archivos_gt_todos)} archivos totales en Drive (se filtraran a los 200 de validacion).")

# No se descarga nada hasta confirmar que ambos modelos tienen las 200
# predicciones de validación; si uno de los dos tiene menos, se ha producido
# un fallo en el entrenamiento o la predicción que debe investigarse
# antes de iniciar la descarga
for trainer, archivos in archivos_val_por_trainer.items():
    assert len(archivos) == 200, (
        f"{trainer} tiene {len(archivos)} predicciones, se esperaban 200. "
        "Revisa el entrenamiento/prediccion antes de continuar."
    )
print("\nConfirmado: 200/200 predicciones para ambos modelos. Se puede descargar.")


nnUNetTrainer_100epochs: 200 predicciones .nii.gz encontradas en Drive.
nnUNetTrainerV1_1000casos_100epochs: 200 predicciones .nii.gz encontradas en Drive.
gt_segmentations: 1000 archivos totales en Drive (se filtraran a los 200 de validacion).

Confirmado: 200/200 predicciones para ambos modelos. Se puede descargar.


## 5. Descarga local de las predicciones y del ground truth necesario

Es la parte más pesada del notebook, aunque se trata de archivos pequeños (máscaras de segmentación, no imágenes de intensidad), por lo que requiere pocos minutos.


In [5]:
for trainer, archivos in archivos_val_por_trainer.items():
    t0 = time.time()
    pend = descargar_lista(archivos, LOCAL_VAL[trainer])
    print(f"{trainer}: {(time.time()-t0)/60:.1f} min, {len(pend)} archivos sin descargar")
    assert not pend, f"Quedaron predicciones sin descargar para {trainer}."

# Solo se requiere el GT de los casos de validación (mismo fold
# para ambos modelos, por lo que basta la unión de ambos conjuntos)
nombres_val = {f["name"] for f in next(iter(archivos_val_por_trainer.values()))}
archivos_gt_val = [f for f in archivos_gt_todos if f["name"] in nombres_val]
print(f"GT necesario para fold {FOLD}: {len(archivos_gt_val)} de {len(archivos_gt_todos)} totales.")

t0 = time.time()
pend_gt = descargar_lista(archivos_gt_val, LOCAL_GT)
print(f"gt_segmentations (fold {FOLD}): {(time.time()-t0)/60:.1f} min, {len(pend_gt)} archivos sin descargar")
assert not pend_gt, "Quedaron archivos de GT sin descargar."

print("\nTodo descargado localmente. La evaluacion ya no necesita tocar Drive.")


Total de archivos a descargar: 200
--- Intento 1/5: 200 archivos pendientes ---
  0/200 procesados (0 fallos hasta ahora)
  50/200 procesados (0 fallos hasta ahora)
  100/200 procesados (0 fallos hasta ahora)
  150/200 procesados (0 fallos hasta ahora)
Descarga completa (verificada por tamano).
nnUNetTrainer_100epochs: 10.7 min, 0 archivos sin descargar
Total de archivos a descargar: 200
--- Intento 1/5: 200 archivos pendientes ---
  0/200 procesados (0 fallos hasta ahora)
  50/200 procesados (0 fallos hasta ahora)
  100/200 procesados (0 fallos hasta ahora)
  150/200 procesados (0 fallos hasta ahora)
Descarga completa (verificada por tamano).
nnUNetTrainerV1_1000casos_100epochs: 10.0 min, 0 archivos sin descargar
GT necesario para fold 1: 200 de 1000 totales.
Total de archivos a descargar: 200
--- Intento 1/5: 200 archivos pendientes ---
  0/200 procesados (0 fallos hasta ahora)
  50/200 procesados (0 fallos hasta ahora)
  100/200 procesados (0 fallos hasta ahora)


  150/200 procesados (0 fallos hasta ahora)
Descarga completa (verificada por tamano).
gt_segmentations (fold 1): 61.1 min, 0 archivos sin descargar

Todo descargado localmente. La evaluacion ya no necesita tocar Drive.


In [7]:
!pip install -q medpy scikit-image SimpleITK

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 46.4 MB/s eta 0:00:00


## 6. Funciones de métricas (idénticas a las de los notebooks 28 y 29 para garantizar la comparabilidad)


In [8]:
import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy import ndimage
from skimage.morphology import skeletonize
from medpy.metric.binary import hd95 as medpy_hd95

CONECTIVIDAD_26 = np.ones((3, 3, 3), dtype=int)

def cargar_volumen(path):
    img = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(img).astype(np.uint8)
    spacing = img.GetSpacing()[::-1]
    return array, spacing

def dice_score(pred, gt):
    inter = np.logical_and(pred, gt).sum()
    return (2.0 * inter) / (pred.sum() + gt.sum() + 1e-8)

def metricas_topologicas(pred, gt):
    """clDice + sus dos componentes por separado (topological precision y
    topological sensitivity), utiles para distinguir si un modelo recupera
    mas centrolinea real (sensibilidad) a costa de introducir mas estructura
    incorrecta (precision), o viceversa -- algo que clDice por si solo oculta.
    """
    skel_pred = skeletonize(pred.astype(bool))
    skel_gt = skeletonize(gt.astype(bool))
    tprec = (skel_pred & gt).sum() / (skel_pred.sum() + 1e-8)
    tsens = (skel_gt & pred).sum() / (skel_gt.sum() + 1e-8)
    cldice = (2 * tprec * tsens) / (tprec + tsens + 1e-8)
    return cldice, tprec, tsens

def componentes_conexas(pred):
    labeled, n = ndimage.label(pred, structure=CONECTIVIDAD_26)
    return labeled, n

def cobertura_componente_principal(pred, gt):
    labeled, n = componentes_conexas(pred)
    if n == 0:
        return 0.0, 0
    tamanos = ndimage.sum(pred, labeled, index=range(1, n + 1))
    idx_mayor = np.argmax(tamanos) + 1
    componente_mayor = labeled == idx_mayor
    cobertura = (componente_mayor & gt.astype(bool)).sum() / (gt.sum() + 1e-8)
    return cobertura, n

def hd95_seguro(pred, gt, spacing):
    if pred.sum() == 0 or gt.sum() == 0:
        return np.nan
    return medpy_hd95(pred, gt, voxelspacing=spacing)

def calcular_metricas_caso(pred_path, gt_path):
    pred, spacing = cargar_volumen(pred_path)
    gt, _ = cargar_volumen(gt_path)
    cobertura, n_componentes = cobertura_componente_principal(pred, gt)
    cldice, topo_precision, topo_sensitivity = metricas_topologicas(pred, gt)
    return {
        "dice": dice_score(pred, gt),
        "cldice": cldice,
        "topo_precision": topo_precision,
        "topo_sensitivity": topo_sensitivity,
        "cobertura_principal": cobertura,
        "n_componentes": n_componentes,
        "hd95_crudo_mm": hd95_seguro(pred, gt, spacing),
    }

print("Funciones de metricas cargadas.")


Funciones de metricas cargadas.


## 7. `evaluar_trainer` con checkpoint (cálculo local y checkpoint en Drive)

El **cálculo** (lectura de NIfTI, Dice, clDice, HD95 y componentes) es completamente local, sin Drive en el bucle pesado. Solo el checkpoint, un CSV pequeño, se escribe directamente en Drive (mediante FUSE, que ha resultado estable para escrituras pequeñas e incrementales; el problema de FUSE se limita a las transferencias masivas de archivos grandes).

El checkpoint se guarda **tras cada caso** (`guardar_cada=1`): al tratarse de CSV diminutos, un fallo en el caso 173 conserva el progreso hasta el caso 172 (y no hasta el 160). Además, el checkpoint sobrevive a la destrucción del runtime, ya que reside en Drive y no en `/content`.


In [9]:
def evaluar_trainer(trainer_name, guardar_cada=1):
    carpeta_val = LOCAL_VAL[trainer_name]
    preds = sorted(carpeta_val.glob("*.nii.gz"))
    if not preds:
        raise RuntimeError(f"No se encontraron predicciones en {carpeta_val}")

    checkpoint_path = BASE / f"checkpoint_fold1_{trainer_name}.csv"

    filas = []
    ya_hechos = set()
    if checkpoint_path.exists():
        df_prev = pd.read_csv(checkpoint_path)
        filas = df_prev.to_dict("records")
        ya_hechos = set(df_prev["case_id"])
        print(f"Reanudando: {len(ya_hechos)} casos ya calculados antes, se saltan.")

    total = len(preds)
    for i, pred_path in enumerate(preds):
        case_id = pred_path.name.replace(".nii.gz", "")
        if case_id in ya_hechos:
            continue

        gt_path = LOCAL_GT / pred_path.name
        if not gt_path.exists():
            continue

        m = calcular_metricas_caso(pred_path, gt_path)
        m["case_id"] = case_id
        filas.append(m)

        if (i + 1) % guardar_cada == 0 or (i + 1) == total:
            pd.DataFrame(filas).to_csv(checkpoint_path, index=False)
            print(f"[{trainer_name}] {i + 1}/{total} procesados -- checkpoint guardado")

    df = pd.DataFrame(filas)
    resumen = {
        "dice_mean": df["dice"].mean(),
        "cldice_mean": df["cldice"].mean(),
        "topo_precision_mean": df["topo_precision"].mean(),
        "topo_sensitivity_mean": df["topo_sensitivity"].mean(),
        "n_componentes_mean": df["n_componentes"].mean(),
        "hd95_crudo_mean": df["hd95_crudo_mm"].mean(skipna=True),
        "n_casos": len(df),
    }
    return df, resumen

print("evaluar_trainer (checkpoint por caso, en Drive) listo.")


evaluar_trainer (checkpoint por caso, en Drive) listo.


## 8. Evaluación de ambos modelos en el fold 1 y guardado de métricas caso a caso (en Drive)


In [10]:
df_base_f1, resumen_base_f1 = evaluar_trainer(BASELINE_TRAINER)
print("=== Baseline (Dice+CE), fold 1, 100 epocas ===")
print(f"Dice: {resumen_base_f1['dice_mean']:.4f} | clDice: {resumen_base_f1['cldice_mean']:.4f} | "
      f"topo_prec: {resumen_base_f1['topo_precision_mean']:.4f} | topo_sens: {resumen_base_f1['topo_sensitivity_mean']:.4f} | "
      f"n_componentes: {resumen_base_f1['n_componentes_mean']:.2f} | HD95: {resumen_base_f1['hd95_crudo_mean']:.2f} mm | "
      f"n={resumen_base_f1['n_casos']}")

csv_base_f1 = BASE / "baseline_1000casos_fold1_100epochs_metricas_por_caso.csv"
df_base_f1.to_csv(csv_base_f1, index=False)
print("Guardado:", csv_base_f1)

df_v1_f1, resumen_v1_f1 = evaluar_trainer(V1_TRAINER)
print("\n=== v1 λ=.2, fold 1, 100 epocas ===")
print(f"Dice: {resumen_v1_f1['dice_mean']:.4f} | clDice: {resumen_v1_f1['cldice_mean']:.4f} | "
      f"topo_prec: {resumen_v1_f1['topo_precision_mean']:.4f} | topo_sens: {resumen_v1_f1['topo_sensitivity_mean']:.4f} | "
      f"n_componentes: {resumen_v1_f1['n_componentes_mean']:.2f} | HD95: {resumen_v1_f1['hd95_crudo_mean']:.2f} mm | "
      f"n={resumen_v1_f1['n_casos']}")

csv_v1_f1 = BASE / "v1_1000casos_fold1_100epochs_metricas_por_caso.csv"
df_v1_f1.to_csv(csv_v1_f1, index=False)
print("Guardado:", csv_v1_f1)


[nnUNetTrainer_100epochs] 1/200 procesados -- checkpoint guardado
[nnUNetTrainer_100epochs] 2/200 procesados -- checkpoint guardado
[nnUNetTrainer_100epochs] 3/200 procesados -- checkpoint guardado
[nnUNetTrainer_100epochs] 4/200 procesados -- checkpoint guardado
[nnUNetTrainer_100epochs] 5/200 procesados -- checkpoint guardado
[nnUNetTrainer_100epochs] 6/200 procesados -- checkpoint guardado
[nnUNetTrainer_100epochs] 7/200 procesados -- checkpoint guardado
[nnUNetTrainer_100epochs] 8/200 procesados -- checkpoint guardado
[nnUNetTrainer_100epochs] 9/200 procesados -- checkpoint guardado
[nnUNetTrainer_100epochs] 10/200 procesados -- checkpoint guardado
[nnUNetTrainer_100epochs] 11/200 procesados -- checkpoint guardado
[nnUNetTrainer_100epochs] 12/200 procesados -- checkpoint guardado
[nnUNetTrainer_100epochs] 13/200 procesados -- checkpoint guardado
[nnUNetTrainer_100epochs] 14/200 procesados -- checkpoint guardado
[nnUNetTrainer_100epochs] 15/200 procesados -- checkpoint guardado
[nnU

## 9. Comparación pareada en el fold 1 (mismos pacientes de validación)

Se prioriza **Wilcoxon**, que no asume normalidad y es más robusto con n = 200 y métricas acotadas como Dice y clDice. Además del p-valor, se reportan el **cambio medio y mediano** de cada métrica, de modo que pueda cuantificarse la magnitud del cambio en cada aspecto de la conectividad (clDice, precisión topológica, sensibilidad topológica, componentes y HD95). El t-test pareado se incluye como referencia secundaria.


In [11]:
from scipy import stats

df_base_f1s = df_base_f1.sort_values("case_id").reset_index(drop=True)
df_v1_f1s = df_v1_f1.sort_values("case_id").reset_index(drop=True)

assert list(df_base_f1s["case_id"]) == list(df_v1_f1s["case_id"]), \
    "case_id no coinciden entre baseline y v1 en fold 1 -- revisa el split."

delta_componentes_f1 = df_base_f1s["n_componentes"] - df_v1_f1s["n_componentes"]

resumen_tabla_f1 = pd.DataFrame([
    {"modelo": "baseline (fold 1)", "dice": resumen_base_f1["dice_mean"], "cldice": resumen_base_f1["cldice_mean"],
     "topo_prec": resumen_base_f1["topo_precision_mean"], "topo_sens": resumen_base_f1["topo_sensitivity_mean"],
     "n_componentes": resumen_base_f1["n_componentes_mean"], "hd95_mm": resumen_base_f1["hd95_crudo_mean"]},
    {"modelo": "v1 λ=.2 (fold 1)", "dice": resumen_v1_f1["dice_mean"], "cldice": resumen_v1_f1["cldice_mean"],
     "topo_prec": resumen_v1_f1["topo_precision_mean"], "topo_sens": resumen_v1_f1["topo_sensitivity_mean"],
     "n_componentes": resumen_v1_f1["n_componentes_mean"], "hd95_mm": resumen_v1_f1["hd95_crudo_mean"]},
])
print(resumen_tabla_f1.to_string(index=False))

print(f"\nΔcomponentes medio (baseline - v1), fold 1 = {delta_componentes_f1.mean():.3f}")
print("(positivo => v1 tiene MENOS componentes que baseline, apoyando H1)")

print("\n=== Tests pareados fold 1 (mismos 200 pacientes): Wilcoxon (principal) + ttest_rel (referencia) ===")
metricas_pareadas = [
    ("dice", "Dice"),
    ("cldice", "clDice"),
    ("topo_precision", "Topological precision"),
    ("topo_sensitivity", "Topological sensitivity"),
    ("n_componentes", "n_componentes"),
    ("hd95_crudo_mm", "HD95"),
]
filas_delta = []
for metrica, nombre in metricas_pareadas:
    a = df_base_f1s[metrica].values   # baseline
    b = df_v1_f1s[metrica].values     # v1
    mask = ~(np.isnan(a) | np.isnan(b))
    a, b = a[mask], b[mask]
    delta = a - b  # baseline - v1
    w_stat, w_p = stats.wilcoxon(a, b)
    t_stat, t_p = stats.ttest_rel(a, b)
    print(f"{nombre}: Δmedio(base-v1)={delta.mean():.4f} | Δmediano={np.median(delta):.4f} | "
          f"Wilcoxon p={w_p:.4g} | ttest_rel p={t_p:.4g} (n={mask.sum()})")
    filas_delta.append({
        "metrica": nombre,
        "media_baseline": a.mean(),
        "media_v1": b.mean(),
        "delta_medio_base_menos_v1": delta.mean(),
        "delta_mediano_base_menos_v1": np.median(delta),
        "wilcoxon_p": w_p,
        "ttest_rel_p": t_p,
        "n": mask.sum(),
    })

df_deltas_f1 = pd.DataFrame(filas_delta)
csv_deltas_f1 = BASE / "tests_pareados_fold1_baseline_vs_v1.csv"
df_deltas_f1.to_csv(csv_deltas_f1, index=False)
print("\nGuardado:", csv_deltas_f1)

csv_delta_f1 = BASE / "delta_componentes_1000casos_fold1_baseline_vs_v1.csv"
pd.DataFrame({
    "case_id": df_base_f1s["case_id"],
    "n_componentes_baseline": df_base_f1s["n_componentes"],
    "n_componentes_v1": df_v1_f1s["n_componentes"],
    "delta_componentes": delta_componentes_f1,
}).to_csv(csv_delta_f1, index=False)
print("Guardado:", csv_delta_f1)


           modelo     dice   cldice  topo_prec  topo_sens  n_componentes   hd95_mm
baseline (fold 1) 0.754104 0.820837   0.765113   0.897870         15.985 22.555574
 v1 λ=.2 (fold 1) 0.766565 0.834807   0.790740   0.893886         13.640 18.703400

Δcomponentes medio (baseline - v1), fold 1 = 2.345
(positivo => v1 tiene MENOS componentes que baseline, apoyando H1)

=== Tests pareados fold 1 (mismos 200 pacientes): Wilcoxon (principal) + ttest_rel (referencia) ===
Dice: Δmedio(base-v1)=-0.0125 | Δmediano=-0.0090 | Wilcoxon p=2.773e-21 | ttest_rel p=1.74e-17 (n=200)
clDice: Δmedio(base-v1)=-0.0140 | Δmediano=-0.0100 | Wilcoxon p=3.219e-18 | ttest_rel p=6.581e-19 (n=200)
Topological precision: Δmedio(base-v1)=-0.0256 | Δmediano=-0.0216 | Wilcoxon p=1.591e-24 | ttest_rel p=1.682e-26 (n=200)
Topological sensitivity: Δmedio(base-v1)=0.0040 | Δmediano=0.0023 | Wilcoxon p=8.262e-05 | ttest_rel p=1.161e-05 (n=200)
n_componentes: Δmedio(base-v1)=2.3450 | Δmediano=2.0000 | Wilcoxon p=2.721e-16 |

## 10. Consistencia entre el fold 0 y el fold 1


In [12]:
df_base_f0 = pd.read_csv(BASE / "baseline_1000casos_100epochs_metricas_por_caso.csv")
df_v1_f0 = pd.read_csv(BASE / "v1_1000casos_100epochs_metricas_por_caso.csv")

def _topo_mean(df, col):
    return df[col].mean() if col in df.columns else np.nan

comparacion_folds = pd.DataFrame([
    {"fold": 0, "modelo": "baseline", "dice": df_base_f0["dice"].mean(), "cldice": df_base_f0["cldice"].mean(),
     "topo_prec": _topo_mean(df_base_f0, "topo_precision"), "topo_sens": _topo_mean(df_base_f0, "topo_sensitivity"),
     "n_componentes": df_base_f0["n_componentes"].mean(), "hd95_mm": df_base_f0["hd95_crudo_mm"].mean()},
    {"fold": 0, "modelo": "v1 λ=.2", "dice": df_v1_f0["dice"].mean(), "cldice": df_v1_f0["cldice"].mean(),
     "topo_prec": _topo_mean(df_v1_f0, "topo_precision"), "topo_sens": _topo_mean(df_v1_f0, "topo_sensitivity"),
     "n_componentes": df_v1_f0["n_componentes"].mean(), "hd95_mm": df_v1_f0["hd95_crudo_mm"].mean()},
    {"fold": 1, "modelo": "baseline", "dice": resumen_base_f1["dice_mean"], "cldice": resumen_base_f1["cldice_mean"],
     "topo_prec": resumen_base_f1["topo_precision_mean"], "topo_sens": resumen_base_f1["topo_sensitivity_mean"],
     "n_componentes": resumen_base_f1["n_componentes_mean"], "hd95_mm": resumen_base_f1["hd95_crudo_mean"]},
    {"fold": 1, "modelo": "v1 λ=.2", "dice": resumen_v1_f1["dice_mean"], "cldice": resumen_v1_f1["cldice_mean"],
     "topo_prec": resumen_v1_f1["topo_precision_mean"], "topo_sens": resumen_v1_f1["topo_sensitivity_mean"],
     "n_componentes": resumen_v1_f1["n_componentes_mean"], "hd95_mm": resumen_v1_f1["hd95_crudo_mean"]},
])
print(comparacion_folds.to_string(index=False))

csv_folds = BASE / "comparacion_fold0_vs_fold1_baseline_vs_v1.csv"
comparacion_folds.to_csv(csv_folds, index=False)
print("\nGuardado:", csv_folds)


 fold   modelo     dice   cldice  topo_prec  topo_sens  n_componentes   hd95_mm
    0 baseline 0.751640 0.813646        NaN        NaN         16.770 23.444308
    0  v1 λ=.2 0.747547 0.811284        NaN        NaN         16.985 24.719474
    1 baseline 0.754104 0.820837   0.765113   0.897870         15.985 22.555574
    1  v1 λ=.2 0.766565 0.834807   0.790740   0.893886         13.640 18.703400

Guardado: /content/drive/MyDrive/TFM_nnUNet/comparacion_fold0_vs_fold1_baseline_vs_v1.csv


## Reanudación

Si el runtime se reinicia durante la sección 8, basta con volver a ejecutar las secciones 1-7 (autenticación, configuración, funciones y descarga, que verifica por tamaño y no repite lo ya descargado) y, a continuación, la sección 8, que se reanuda desde el checkpoint.
